<a href="https://colab.research.google.com/github/jivaniaadit/factor_xa_cheminformatics/blob/main/notebooks/w2_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rdkit

import numpy as np
import pandas as pd
import joblib
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/drugproj'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# trained Random Forest, loaded from disk
final_model = joblib.load(f'{DRIVE}/factor_xa_rf_final.joblib')
print("Model loaded:", final_model.n_estimators, "trees")

# rebuild training fingerprint matrix X for the applicability domain check
df_clean = pd.read_csv(f'{DRIVE}/factor_xa_clustered.csv')
mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

mols = [Chem.MolFromSmiles(s) for s in df_clean['canonical_smiles_std']]
X = np.zeros((len(mols), 2048), dtype=np.int8)
for i, m in enumerate(mols):
    DataStructs.ConvertToNumpyArray(mfpgen.GetFingerprint(m), X[i])

# precompute training bit counts once, used by the fast similarity check
X_int = X.astype(np.int32)
train_counts = X_int.sum(axis=1)
print("Training matrix ready:", X.shape)

Model loaded: 300 trees
Training matrix ready: (3477, 2048)


In [ ]:
def predict_pki(smiles, top_k=5):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {"error": f"could not parse SMILES: {smiles}"}

    fp = mfpgen.GetFingerprint(mol)
    q = np.zeros(2048, dtype=np.int32)
    DataStructs.ConvertToNumpyArray(fp, q)

    pred = final_model.predict(q.reshape(1, -1))[0]

    # vectorized Tanimoto, integer matmul counts shared on-bits correctly
    intersection = X_int @ q                      # shared bits per training row
    union = train_counts + q.sum() - intersection
    sims = np.divide(intersection, union,
                     out=np.zeros(len(union), dtype=float),
                     where=union > 0)

    max_sim = float(sims.max())
    mean_top_k = float(np.sort(sims)[::-1][:top_k].mean())

    if max_sim >= 0.6:
        confidence = "high (close to training data)"
    elif max_sim >= 0.4:
        confidence = "moderate"
    else:
        confidence = "low (out of domain, prediction unreliable)"

    return {
        "smiles": smiles,
        "predicted_pKi": round(float(pred), 2),
        "max_similarity_to_training": round(max_sim, 3),
        "mean_top5_similarity": round(mean_top_k, 3),
        "confidence": confidence,
    }

In [ ]:
demo = [
    ("in-domain (from dataset)", df_clean['canonical_smiles_std'].iloc[0]),
    ("novel analog (moderate)", "O=C(Nc1ccccc1C(=O)Nc1ccccn1)c1cc(CNC(=O)c2ccc(Cl)s2)cs1"),
    ("caffeine (out of domain)", "Cn1cnc2c1c(=O)n(C)c(=O)n2C"),
]

import pandas as pd
rows = []
for label, smi in demo:
    r = predict_pki(smi)
    r['label'] = label
    rows.append(r)

demo_df = pd.DataFrame(rows)[['label', 'predicted_pKi',
                              'max_similarity_to_training', 'confidence']]
print(demo_df.to_string(index=False))

                   label  predicted_pKi  max_similarity_to_training                                 confidence
in-domain (from dataset)           8.90                       1.000              high (close to training data)
 novel analog (moderate)           7.37                       0.427                                   moderate
caffeine (out of domain)           6.56                       0.171 low (out of domain, prediction unreliable)


In [ ]:
print(predict_pki('O=C(Nc1ccccc1C(=O)Nc1ccccn1)c1cc(CNC(=O)c2ccc(Cl)s2)cs1'))

{'smiles': 'O=C(Nc1ccccc1C(=O)Nc1ccccn1)c1cc(CNC(=O)c2ccc(Cl)s2)cs1', 'predicted_pKi': 7.37, 'max_similarity_to_training': 0.427, 'mean_top5_similarity': 0.375, 'confidence': 'moderate'}


In [ ]:
print(predict_pki('COC1=CC=C(C=C1)N2C3=C(CCN(C3=O)C4=CC=C(C=C4)N5CCCCC5=O)C(=N2)C(=O)N'))

{'smiles': 'COC1=CC=C(C=C1)N2C3=C(CCN(C3=O)C4=CC=C(C=C4)N5CCCCC5=O)C(=N2)C(=O)N', 'predicted_pKi': 9.75, 'max_similarity_to_training': 1.0, 'mean_top5_similarity': 0.784, 'confidence': 'high (close to training data)'}


In [ ]:
print(predict_pki('O=C(C1=CC=C(Cl)S1)N2CCN(C3=CC=C(N4CCOCC4)C=C3)CC2'))

{'smiles': 'O=C(C1=CC=C(Cl)S1)N2CCN(C3=CC=C(N4CCOCC4)C=C3)CC2', 'predicted_pKi': 7.67, 'max_similarity_to_training': 0.413, 'mean_top5_similarity': 0.371, 'confidence': 'moderate'}
